# Gold standard v2, por design

Constrói a base de designs elegíveis, sorteia 36 e cruza cada um com o gold standard v1, o pool das 572 famílias e os 626 designs do dataset.
Não corre nada no PatSeer: as queries ficam escritas em `gs_v2_queries.txt`.

Saídas em `06_gold_standard/v2/`: `eligible.csv`, `gs_v2_sample.csv`, `gs_v2_draw_order.csv`, `gs_v2_index_map.csv`, `gs_v2_queries.txt`.

**Decisões de construção que não vêm do pedido** (rever):
1. `03_aircraft/trl/` só tem TRL para designs patenteados do dataset. Para as outras páginas VFS o TRL é um piso tirado do estado da página (`trl_fonte = estado VFS`).
2. O "ano do 1.º voo" é o `final_year` de `YEAR.csv` (ano de desenvolvimento), ou o `first_flight` do índice AAM quando existe.
3. Um design = empresa + modelo, depois de juntar as páginas VFS do mesmo modelo.
4. "Empresa com nome legal" = campo empresa preenchido e fora da lista `NOT_LEGAL`.


In [ ]:
from pathlib import Path
import re, numpy as np, pandas as pd
pd.set_option("display.width", 250, "display.max_colwidth", 60, "display.max_rows", 300)

DATA_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path("eVTOL_dataset_v1")
assert (DATA_DIR / "05_secondary/vfs/aircraft.csv").exists(), DATA_DIR
OUT = DATA_DIR / "06_gold_standard/v2"
SEED = 42
# quotas: Prompt 3 pedia 10/10/10/3/3; o autor passou a amostra para 68 em 2026-11-10 (a ordem de sorteio é a mesma, só se desce mais)
QUOTA = [("VT", "Vectored Thrust", 20), ("LC", "Lift + Cruise", 20), ("WM", "Wingless", 20), ("ER", "Rotorcraft", 3), ("HB", "Hoverbike/PFD", 5)]
CAT = {k: v for k, v, _ in QUOTA}
print(DATA_DIR)

## 0. Chaves de empresa e de modelo

In [ ]:
# primeira regra que acerta dá a chave; serve para cruzar VFS, índice, pool, 626 e gold v1
ALIAS = [("ehang", "ehang"), ("volocopter|e-volo", "volocopter"), ("joby", "joby"), ("beta (tech|air)|^beta$", "beta"),
         ("archer", "archer"), ("wisk", "wisk"), ("kitty ?hawk|zee\\.?aero", "kittyhawk"), ("embraer|\\beve\\b", "eve"),
         ("aerofugia|wofei|geely", "aerofugia"), ("xpeng|aridge|huitian", "xpeng"), ("hyundai|supernal", "supernal"),
         ("bell|textron", "bell"), ("airbus", "airbus"), ("honda", "honda"), ("autoflight", "autoflight"),
         ("skydrive|cartivator", "skydrive"), ("vertical aerospace", "verticalaerospace"), ("lift aircraft", "liftaircraft"),
         ("lilium", "lilium"), ("ascendance", "ascendance"), ("volant|wollant", "volant"), ("tetra", "tetra"),
         ("trek aerospace", "trekaerospace"), ("avic|chrdi|aviation industry corporation of china", "avic"),
         ("pivotal|skykar|opener", "pivotal"), ("zero ?g|zerog", "zerog"), ("tcab", "tcab"), ("vertaxi", "vertaxi"),
         ("erc system", "ercsystem"), ("overair", "overair"), ("zeva|alpine vertical", "zeva"), ("comac|commercial aircraft corporation of china", "comac"), ("tier 1 engineering", "tier1engineering"), ("moller|freedom motors", "moller")]
LEGAL = r"\b(inc|llc|ltd|limited|gmbh|co|corp|corporation|company|se|sas|sa|ag|ab|jsc|pte|pvt|plc|srl|bv)\b\.?"
def ckey(s):
    if not isinstance(s, str) or not s.strip(): return ""
    t = s.lower()
    for pat, key in ALIAS:
        if re.search(pat, t): return key
    t = re.sub(r"\(.*?\)", " ", t); t = re.sub(LEGAL, " ", t)
    return re.sub(r"[^a-z0-9]+", "", t)

MODEL_ALIAS = {("airbus", "vahanaalphaone"): "vahana", ("airbus", "vahanaalphatwo"): "vahana", ("airbus", "vahanabeta"): "vahana",
               ("overair", "butterflyv1"): "butterfly", ("overair", "butterflyv2"): "butterfly",
               ("lilium", "liliumphoenix"): "liliumjet",   # Phoenix = demonstradores do Lilium Jet
               ("tier1engineering", "er441stgeneration"): "er44", ("tier1engineering", "er442ndgeneration"): "er44", ("tier1engineering", "er443rdgeneration"): "er44",
               ("ehang", "eh216f"): "eh216s", ("ehang", "eh216l"): "eh216s",   # mesma célula, variantes de missão
               ("joby", "s420"): "s4", ("joby", "s410"): "s4", ("joby", "s4t"): "s4",
               ("eve", "eveunnamed"): "eve", ("eve", "evev3"): "eve",
               ("autoflight", "prosperty1v1500m"): "prosperityv2000em",
               ("wisk", "generation5cora"): "cora", ("wisk", "generation4cora"): "cora"}
def mkey(ck, model):
    m = re.sub(r"[^a-z0-9]+", "", str(model).lower())
    return MODEL_ALIAS.get((ck, m), m)

# empresas sem nome legal: pessoas, projectos, equipas sem entidade
NOT_LEGAL = r"^(.*diy.*|alan zhang|razvan sabie|tim merk|project .*|.*\(original company name\).*|unknown.*|individual.*|private.*|anonymous.*)$"

## 1. Base de elegíveis
### 1a. VFS: páginas, domínio, ano e TRL

In [ ]:
a  = pd.read_csv(DATA_DIR / "05_secondary/vfs/aircraft.csv")
d  = pd.read_csv(DATA_DIR / "05_secondary/vfs/DOMAIN.csv")
y  = pd.read_csv(DATA_DIR / "05_secondary/vfs/YEAR.csv")
dy = pd.read_csv(DATA_DIR / "05_secondary/vfs/aircraft_dev_year.csv")
pl = pd.read_csv(DATA_DIR / "05_secondary/vfs/patent_links.csv")
trl = pd.read_csv(DATA_DIR / "03_aircraft/trl/TRL.csv")
a627 = pd.read_csv(DATA_DIR / "03_aircraft/aircraft_627.csv", dtype=str)

v = (a[["slug_key", "url", "list_name", "model", "company", "status", "evtol_class_main", "n_images", "spec_aircraft_type"]]
     .merge(d[["slug_key", "status_group", "in_domain"]], on="slug_key")
     .merge(y[["slug_key", "final_year"]], on="slug_key", how="left")
     .merge(dy[["slug_key", "basis_a"]], on="slug_key", how="left"))

# TRL revisto (03_aircraft/trl) através das ligações página VFS -> design patenteado
pl = pl[pl.use == "yes"].merge(trl[["aircraft_id", "final_trl"]], left_on="aircraft_uid", right_on="aircraft_id")
slug_trl, slug_ids = {}, {}
for _, r in pl.iterrows():
    for s in str(r.slug_keys).split("|"):
        slug_trl[s] = max(slug_trl.get(s, 0), r.final_trl); slug_ids.setdefault(s, set()).add(r.aircraft_uid)
v["trl_dataset"] = v.slug_key.map(slug_trl)
# TRL do script de keywords da tese corrido em todas as páginas (notebooks/02a_vfs_page_trl.py). Só informativo:
# nas páginas fora do dataset dá TRL >= 5 a 86 % do domínio (232 de 274 páginas "concept design"), por isso não filtra.
ps = pd.read_csv(DATA_DIR / "05_secondary/vfs/VFS_PAGE_TRL_SCRIPT.csv")
v = v.merge(ps[["slug_key", "trl_script"]], on="slug_key", how="left")
# TRL >= 7 por regra de palavras-chave (notebooks/02c_vfs_page_trl7.py): voo tripulado, transição para voo em frente, ou certificado
p7 = pd.read_csv(DATA_DIR / "05_secondary/vfs/VFS_PAGE_TRL7.csv").rename(columns={"sentence": "frase_trl7"})
v = v.merge(p7, on="slug_key", how="left")

# piso de TRL pelo estado da página, segundo a regra de trl_scale_nasa.csv
def status_trl(s):
    s = str(s).lower()
    if re.search(r"subscale|proof.of.concept", s): return 5
    if re.search(r"production|prototype|technology demonstrator", s): return 6
    return np.nan
v["trl_estado"] = np.where(v.status_group == "flying hardware", v.status.map(status_trl), np.nan)
# Rotorcraft: quase nenhuma página tem estado, por isso as 57 páginas do domínio foram lidas (gs_v2_rotorcraft_decisions.csv).
# Vale a coluna `voou` do autor quando preenchida; senão a proposta do Claude.
rd = pd.read_csv(OUT / "gs_v2_rotorcraft_decisions.csv", dtype=str).fillna("")
rd["dec"] = np.where(rd.voou != "", rd.voou, rd.proposta_claude)
rd["quem"] = np.where(rd.voou != "", "leitura da página (autor)", "leitura da página (proposta Claude, por confirmar)")
ok = rd[rd.dec == "s"].set_index("slug_key")
v["trl_leitura"] = pd.to_numeric(v.slug_key.map(ok.trl_leitura), errors="coerce")
v["trl"] = v.trl_dataset.fillna(v.trl_estado).fillna(v.trl_leitura)          # o TRL revisto manda sobre o estado
v["trl_fonte"] = np.where(v.trl_dataset.notna(), "03_aircraft/trl", np.where(v.trl_estado.notna(), "estado VFS", np.where(v.trl_leitura.notna(), v.slug_key.map(ok.quem), "")))
v.loc[v.slug_key.isin(rd[rd.dec == "n"].slug_key), ["trl", "trl_fonte"]] = [np.nan, ""]
v["ck"] = v.company.map(ckey); v["mk"] = [mkey(c, m) for c, m in zip(v.ck, v.model)]
print(len(v), "páginas |", int(v.in_domain.sum()), "no domínio |", int((v.in_domain & (v.trl >= 5)).sum()), "no domínio com TRL >= 5")
print(pd.crosstab(v[v.in_domain].trl_fonte.replace("", "sem TRL"), v[v.in_domain].trl.fillna(0).astype(int)))

### 1b. Uma linha por design (empresa + modelo)

In [ ]:
# carros voadores (rodoviários) não entram no dataset (regra do autor, 2026-11-09/10). "flying car" sozinho não conta:
# o VFS usa-o como termo genérico; conta o tipo de aeronave da ficha dizer que anda na estrada.
CARRO = re.compile(r"roadable|road[- ]legal|air ?/ ?ground|air[- ]ground|ground vehicle|flying and driving", re.I)
RANK = {"production": 4, "prototype": 3, "technology demonstrator": 3, "subscale": 2, "proof": 2}
def page_rank(s):
    s = str(s).lower()
    return max([r for k, r in RANK.items() if k in s] + [0])
v["prank"] = v.status.map(page_rank)

def collapse(g):
    best = g.sort_values(["prank", "n_images"], ascending=False).iloc[0]
    ids = sorted(set().union(*[slug_ids.get(s, set()) for s in g.slug_key]))
    yr = g.loc[g.prank == g.prank.max(), "final_year"].min()
    if pd.isna(yr): yr = g.final_year.min()
    return pd.Series({"empresa": best.company, "aeronave": best.model, "classe": g.evtol_class_main.mode().iloc[0],
                      "trl": g.trl.max(), "trl_script": g.trl_script.max(), "trl7": g.trl7.max(),
                      "frase_trl7": (g.sort_values("trl7", ascending=False).frase_trl7.fillna("").iloc[0]), "trl_fonte": "; ".join(sorted(set(g.trl_fonte) - {""})),
                      "ano_vfs": yr, "ano_base": best.basis_a, "dominio_vfs": bool(g.in_domain.any()),
                      "tem_imagem": bool((g.n_images > 0).any()),
                      "carro_voador": bool(g.spec_aircraft_type.fillna("").str.contains(CARRO).any()),
                      "tipo_vfs": " | ".join(sorted(set(g.spec_aircraft_type.fillna("")) - {""}))[:200], "n_paginas": len(g), "estado_vfs": best.status,
                      "slugs": "|".join(g.slug_key), "url": best.url, "ids_dataset": "; ".join(ids)})
des = v[v.ck != ""].groupby(["ck", "mk"]).apply(collapse, include_groups=False).reset_index()
des["nome_legal"] = ~des.empresa.str.lower().str.strip().str.match(NOT_LEGAL)
print(len(des), "designs VFS |", int(des.dominio_vfs.sum()), "no domínio")

### 1c. Índice AAM (24 empresas), ligado às páginas VFS

In [ ]:
ari = pd.read_csv(DATA_DIR / "05_secondary/aam_index/ari_may2026.csv")
cmap = pd.read_csv(DATA_DIR / "05_secondary/aam_index/COMPANY_MAP.csv")
ari["oem_clean"] = ari.oem.str.replace(r"\s*\(.*?\)", "", regex=True).str.strip()
ari = ari.merge(cmap[["oem", "company_canonical", "domain_flag"]], left_on="oem_clean", right_on="oem", how="left", suffixes=("", "_map"))
assert len(ari) == 24

# veículo do índice -> modelo VFS (chave mk); None = sem página VFS
INDEX_VEHICLE = {("ehang", "EH216-S/F"): ["eh216s"], ("ehang", "VT-35"): ["vt35"],
    ("volocopter", "VoloCity"): ["volocity"], ("volocopter", "VoloUrban"): None, ("volocopter", "VoloRegion"): ["voloregion"],
    ("volant", "VE25-100 SKYLAR"): ["ve25100"], ("joby", "S4"): ["s4"], ("beta", "CX300"): None, ("beta", "A250"): ["alia250"],
    ("autoflight", "V2000EM Prosperity"): ["prosperityv2000em"], ("archer", "Midnight"): ["midnight"], ("eve", "Eve"): ["eve"],
    ("wisk", "Generation 6"): ["generation6"], ("aerofugia", "AE200-100"): ["ae200x01"], ("skydrive", "SKYDRIVE"): ["sd05"],
    ("verticalaerospace", "Valo"): ["vx4"], ("tcab", "E20"): ["e20"], ("zerog", "ZG-ONE"): ["zgone"], ("zerog", "ZG-T6"): ["zgt6"],
    ("ascendance", "Atea"): ["atea"], ("honda", "F1"): ["f1"], ("vertaxi", "M1"): ["matrix1"], ("ercsystem", "-"): ["romeo"],
    ("supernal", "S-A2"): ["sa2"]}
INDEX_NOTE = {("verticalaerospace", "Valo"): "o índice nomeia o Valo (versão de produção); a página VFS é a do VX4",
              ("ercsystem", "-"): "o índice não nomeia o veículo; ligado ao Romeo (lift + cruise)",
              ("skydrive", "SKYDRIVE"): "ligado ao SD-05", ("aerofugia", "AE200-100"): "ligado ao AE200 X01"}
rows = []
for _, r in ari.iterrows():
    ck = ckey(r.oem_clean)
    veh = [x.strip() for x in re.split(r"\s/\s|/\s", str(r.vehicles))]
    ff = [x.strip() for x in str(r.first_flight).split("/")]
    for i, name in enumerate(veh):
        year = pd.to_numeric(ff[i] if i < len(ff) else ff[-1], errors="coerce")
        out = isinstance(r.domain_flag, str) and r.domain_flag.startswith("out")
        mks = None if out else INDEX_VEHICLE.get((ck, name))
        note = "fora: não é VTOL (COMPANY_MAP.domain_flag)" if out else INDEX_NOTE.get((ck, name), "" if mks else "sem página VFS")
        for mk in (mks or [None]):
            rows.append({"oem": r.oem_clean, "ck": ck, "veiculo_indice": name, "ano_indice": year, "mk": mk, "nota": note})
imap = pd.DataFrame(rows)
imap = imap.merge(des[["ck", "mk", "aeronave", "classe", "dominio_vfs"]], on=["ck", "mk"], how="left")
imap.to_csv(OUT / "gs_v2_index_map.csv", index=False)
print(imap.oem.nunique(), "empresas |", int(imap.aeronave.notna().sum()), "designs com página VFS |", int(imap.aeronave.isna().sum()), "veículos sem página")
imap

### 1d. Os 11 designs em falta vindos das 51 famílias do pool

In [ ]:
rv = pd.read_excel(OUT / "goldstandard_pool_review.xlsx")
p = rv[rv.priority_group.notna() & (rv.coder_in_domain == "yes") & (rv.coder_shows_crewed_aircraft == "s") & (rv.coder_in_627 == "n")].copy()
p["dk"] = p.coder_design_key.fillna(p["Record Number"])
p["yr"] = pd.to_numeric(p["Priority Year"], errors="coerce")
pool = p.groupby("dk").agg(familias=("Record Number", lambda s: "; ".join(sorted(s))), asn=("asn_company", "first"),
                           assignee=("Current Assignee", "first"), arch=("coder_architecture", "first"),
                           vfs=("coder_vfs_design", "first"), grupo=("priority_group", lambda s: s.iloc[0][:1]), ano=("yr", "min")).reset_index()
assert len(pool) == 11 and (pool.arch == "MR").sum() == 8, pool
# design do pool -> design VFS, só onde o nome foi confirmado na codificação
POOL_VFS = {"CN214267937U": ("ehang", "eh216s"), "US11487283B1": ("liftaircraft", "hexa"), "WO2024176796A1": ("skydrive", "sd05"),
            "US2015012154A1": ("volocopter", "vc200"), "volocopter_DE102018133096A1": ("volocopter", "2x"),
            "huitian_CN220996738U": ("xpeng", "landaircraftcarrier")}
pool["ck"] = [POOL_VFS.get(k, (ckey(a_), None))[0] for k, a_ in zip(pool.dk, pool.asn)]
pool["mk"] = [POOL_VFS.get(k, (None, "pool_" + k))[1] for k in pool.dk]
chk = pool.merge(des[["ck", "mk", "aeronave"]], on=["ck", "mk"], how="left")
print(chk[["dk", "asn", "arch", "vfs", "ck", "mk", "aeronave", "ano"]].to_string())

### 1e. Elegibilidade e `eligible.csv`

In [ ]:
idx = imap.dropna(subset=["aeronave"]).groupby(["ck", "mk"]).agg(ano_indice=("ano_indice", "min"), veiculo_indice=("veiculo_indice", "first"), nota_indice=("nota", "first")).reset_index()
e = des.merge(idx, on=["ck", "mk"], how="left")
e["via_vfs"] = e.dominio_vfs & (e.trl >= 5)
e["via_indice"] = e.veiculo_indice.notna()
# pool
e = e.merge(pool[["ck", "mk", "familias", "arch"]].rename(columns={"familias": "familia_pool", "arch": "arch_pool"}), on=["ck", "mk"], how="left")
only = pool[~pool.set_index(["ck", "mk"]).index.isin(e.set_index(["ck", "mk"]).index)]
extra = pd.DataFrame({"ck": only.ck, "mk": only.mk, "empresa": only.assignee.str.split(";").str[0].str.replace(r"\s*\(.*?\)", "", regex=True).str.strip(),
    "aeronave": ["(sem nome) " + f.split(";")[0] for f in only.familias],
    "classe": only.arch.map(lambda x: "WM" if x == "MR" else "VT"), "ano_vfs": only.ano, "ano_base": "ano de prioridade da família",
    "familia_pool": only.familias, "arch_pool": only.arch, "nome_legal": True, "dominio_vfs": False, "tem_imagem": False,
    "via_vfs": False, "via_indice": False, "trl_fonte": "", "nota_pool": only.vfs.fillna("").map(lambda s: ("parecido no VFS: " + s) if s else "")})
e = pd.concat([e, extra], ignore_index=True)
e["via_pool"] = e.familia_pool.notna()
e[["via_vfs", "via_indice"]] = e[["via_vfs", "via_indice"]].fillna(False).astype(bool)

base = np.select([e.via_vfs & e.via_indice, e.via_vfs, e.via_indice], ["ambos", "VFS", "índice"], "")
e["origem"] = [("pool" if not b else b + "+pool") if p_ else b for b, p_ in zip(base, e.via_pool)]
# critérios comuns: categoria VFS, nome legal, página com foto ou render (os designs só do pool entram por instrução)
e["elegivel"] = (e.origem != "") & e.classe.isin(CAT) & e.nome_legal.astype(bool) & (e.tem_imagem.astype(bool) | (e.origem == "pool"))
e["ano_1o_voo"] = e.ano_indice.fillna(e.ano_vfs)
e["ano_fonte"] = np.where(e.ano_indice.notna(), "índice AAM (first_flight)", e.ano_base.fillna(""))
e["categoria_vfs"] = e.classe.map(CAT)

print("candidatos por origem, antes dos critérios comuns:"); print(e[e.origem != ""].origem.value_counts().to_string())
drop = e[(e.origem != "") & ~e.elegivel]
print("\nexcluídos pelos critérios comuns:", len(drop)); print(drop[["empresa", "aeronave", "classe", "nome_legal", "tem_imagem", "origem"]].to_string())

cols = ["empresa", "aeronave", "categoria_vfs", "carro_voador", "tipo_vfs", "trl", "trl_fonte", "trl_script", "ano_1o_voo", "ano_fonte", "origem", "familia_pool", "estado_vfs",
        "dominio_vfs", "n_paginas", "url", "slugs", "ids_dataset", "veiculo_indice", "nota_indice", "nota_pool", "ck", "mk", "classe"]
elig = e[e.elegivel].sort_values(["classe", "ck", "mk"]).reset_index(drop=True)[cols]
for c in ["trl", "trl_script", "ano_1o_voo"]: elig[c] = elig[c].astype("Int64")
elig.to_csv(OUT / "eligible.csv", index=False)
print("\neligible.csv:", len(elig), "designs,", elig.ck.nunique(), "empresas")

In [ ]:
order = [v_ for _, v_, _ in QUOTA]
tab = pd.crosstab(elig.categoria_vfs, elig.origem.str.replace(r"\+pool", "", regex=True)).reindex(order).fillna(0).astype(int)
tab["dos quais do pool"] = elig[elig.origem.str.contains("pool")].categoria_vfs.value_counts().reindex(order).fillna(0).astype(int)
tab.insert(0, "elegíveis", elig.categoria_vfs.value_counts().reindex(order).fillna(0).astype(int))
tab["quota"] = [q for _, _, q in QUOTA]
tab.loc["Total"] = tab.sum()
tab

## 2. Sorteio (semente 42)
Dentro de cada categoria os elegíveis são ordenados por empresa e modelo, baralhados, e ficam os primeiros da quota. A ordem completa fica em `gs_v2_draw_order.csv` para as substituições.

In [ ]:
# Âmbito (autor, 2026-11-10): o gold standard mede a completude da query sobre designs PATENTEÁVEIS de programas comerciais.
# Saem da base, como substituição na ordem de sorteio (o sorteio não muda):
#  - actores académicos (universidades, laboratórios, academias), que raramente patenteiam o protótipo;
#  - concurso GoFly (index.csv, coluna gofly), protótipos de concurso;
#  - designs com primeiro ano antes de 2000, fora da janela de publicação do dataset (a família mais antiga das 1 639 é de 2000).
ACAD = re.compile(r"universit|college|academy|laborator|\blab\b|school|student|\bteam\b|institute of technology|caltech", re.I)
gofly_slugs = set(pd.read_csv(DATA_DIR / "05_secondary/vfs/index.csv").query("gofly == True").slug_key)
def ambito(r):
    if ACAD.search(str(r.empresa)): return "fora do âmbito: actor académico (universidade / laboratório), não patenteia o protótipo"
    if any(x in gofly_slugs for x in str(r.slugs).split("|")): return "fora do âmbito: protótipo do concurso GoFly"
    if pd.notna(r.ano_1o_voo) and r.ano_1o_voo < 2000: return "fora do âmbito: anterior a 2000, fora da janela de publicação do dataset"
    return ""
SUBST = {("honda", "f1"): "híbrido: pela regra do autor é 'similar', fora do domínio (decisão do autor, 2026-11-10)",
         ("xpeng", "x3"): "carro voador: não entra no dataset (decisão do autor, 2026-11-10)"}
rng = np.random.default_rng(SEED)
draw = []
for k, name, q in QUOTA:
    g = elig[elig.classe == k].sort_values(["ck", "mk"]).reset_index(drop=True)
    # Rotorcraft tem fluxo próprio: a sua base mudou com a leitura das páginas, e assim as outras categorias mantêm o sorteio já mostrado
    perm = np.random.default_rng([SEED, 1]).permutation(len(g)) if k == "ER" else rng.permutation(len(g))
    if k == "ER": rng.permutation(1)          # o primeiro sorteio tinha 1 Rotorcraft elegível; mantém o estado do gerador
    g = g.iloc[perm].reset_index(drop=True)
    g.insert(0, "ordem_sorteio", range(1, len(g) + 1))
    # substituições decididas pelo autor: o design sai e entra o seguinte na ordem de sorteio
    g["motivo_substituicao"] = [SUBST.get((r.ck, r.mk), "carro voador (ficha VFS: anda na estrada), não entra no dataset (regra do autor)" if r.carro_voador else ambito(r))
                                for r in g.assign(carro_voador=g.carro_voador.fillna(False).astype(bool)).itertuples()]
    g["substituido_por"] = ""; g["substitui"] = ""; g["sorteado"] = False
    need_, pending = q, []
    for i in g.index:
        if need_ == 0: break
        g.loc[i, "sorteado"] = True
        if g.loc[i, "motivo_substituicao"]:
            pending.append(i)
        else:
            need_ -= 1
            if pending and g.loc[i, "ordem_sorteio"] > q:
                j = pending.pop(0); g.loc[j, "substituido_por"] = g.loc[i, "aeronave"]; g.loc[i, "substitui"] = g.loc[j, "aeronave"]
    draw.append(g)
    if len(g) < q: print(f"{name}: só {len(g)} elegíveis para uma quota de {q} -> censo")
draw = pd.concat(draw, ignore_index=True)
draw.drop(columns=["classe"]).to_csv(OUT / "gs_v2_draw_order.csv", index=False)
S = draw[draw.sorteado].drop(columns="sorteado").reset_index(drop=True)
ACT = S.motivo_substituicao == ""
print(len(S), "linhas,", int(ACT.sum()), "designs activos:", S[ACT].categoria_vfs.value_counts().reindex(order).to_dict())

## 3. Cruzamento com o gold v1, o pool das 572 e os 626 designs

In [ ]:
# the thesis gold standard (v1) is only a cross-reference column of the 2026-11-09 sample; since 2026-11-21 it is outside the
# dataset (eVTOL_dataset_v1_WORK/06_gold_standard_v1/), and the two gold_v1 columns stay empty when it is absent
_g1 = DATA_DIR / "06_gold_standard/v1/gold_standard_v1.csv"
gold = pd.read_csv(_g1) if _g1.exists() else pd.DataFrame(columns=["assignee", "design id", "publication number"])
gold["ck"] = gold.assignee.map(lambda s: ckey(str(s).split(";")[0]) if isinstance(s, str) else "")
gold["ids"] = gold["design id"].fillna("").map(lambda s: {x.strip() for x in s.split(";") if x.strip()})
a627["ck"] = a627.company.map(ckey)
a627["nk"] = a627.aircraft_name.fillna("").str.lower().str.replace(r"[^a-z0-9]+", "", regex=True)
ids626 = set(a627.aircraft_id)
rv["ck"] = rv.asn_company.map(ckey)
pool_txt = (rv["Current Assignee"].fillna("") + " | " + rv["Assignee"].fillna("")).str.lower()

def cross(r):
    # designs dos 626: pela ligação página VFS -> patente, ou por empresa + nome
    ids = {x for x in str(r.ids_dataset).split("; ") if x in ids626} if isinstance(r.ids_dataset, str) else set()
    how = "ligação VFS" if ids else ""
    if not ids and not str(r.mk).startswith("pool_"):
        m = a627[(a627.ck == r.ck) & a627.nk.str.startswith(r.mk) & (a627.name_is_real.astype(str).str.lower() == "true")]
        ids, how = set(m.aircraft_id), ("empresa + nome" if len(m) else "")
    g_des = gold[gold.ids.map(lambda s: bool(s & ids))]
    g_co = gold[gold.ck == r.ck]
    n_pool = int((rv.ck == r.ck).sum())
    if n_pool == 0:                                   # empresas fora do mapa asn_company: procura no texto do assignee
        tok = re.sub(r"\(.*?\)", " ", str(r.empresa).lower()); tok = re.sub(LEGAL, " ", tok).strip(" ,.")
        if len(tok) >= 4: n_pool = int(pool_txt.str.contains(re.escape(tok)).sum())
    return pd.Series({"gold_v1_patentes_do_design": "; ".join(g_des["publication number"]),
                      "gold_v1_patentes_da_empresa": len(g_co), "empresa_no_pool_572": "s" if n_pool else "n",
                      "familias_da_empresa_no_pool": n_pool, "design_nos_626": "s" if ids else "n",
                      "ids_nos_626": "; ".join(sorted(ids)), "ligacao_626": how})
S = pd.concat([S, S.apply(cross, axis=1)], axis=1)
# procura nova só quando a empresa não está no pool E o design ainda não está nos 626 (um design já no dataset está capturado)
S["precisa_procura_nova"] = np.where((S.empresa_no_pool_572 == "n") & (S.design_nos_626 == "n") & (S.motivo_substituicao == ""), "s", "n")
S["o_que_falta"] = np.select([S.motivo_substituicao != "", S.design_nos_626 == "s", S.familia_pool.notna(), S.empresa_no_pool_572 == "s"],
                             ["substituído", "nada: já nos 626", "nada: família das 51", "ver famílias da empresa no pool"], "query nova")
out_cols = ["categoria_vfs", "ordem_sorteio", "empresa", "aeronave", "trl", "trl_fonte", "trl_script", "ano_1o_voo", "ano_fonte", "origem",
            "gold_v1_patentes_do_design", "gold_v1_patentes_da_empresa", "empresa_no_pool_572", "familias_da_empresa_no_pool",
            "design_nos_626", "ids_nos_626", "ligacao_626", "familia_pool", "precisa_procura_nova", "o_que_falta",
            "substituido_por", "motivo_substituicao", "substitui", "estado_vfs", "url", "nota_indice", "nota_pool"]
# respostas do autor na página sheets_v2 (famílias do pool), export mais recente; "dúvida" conta como mostrado
# (regra do autor, 2026-11-10: semelhante o bastante para ser da mesma empresa e família). Em branco no export = nenhuma.
ans_f = sorted(OUT.glob("gs_v2_pool_answers_*.csv"))
S["familia_pool_resposta"] = ""; S["resposta_pool"] = ""
# página sheets_v2: quantas famílias candidatas teve cada design (escrito por notebooks/02b_pool_candidates.py)
pdz = OUT / "gs_v2_pool_designs.csv"
ncand = pd.read_csv(pdz).set_index("aeronave").n_familias_candidatas if pdz.exists() else pd.Series(dtype=int)
S["n_familias_candidatas"] = S.aeronave.map(ncand)
if ans_f:
    an = pd.read_csv(ans_f[-1], dtype=str).fillna("").set_index("aeronave")
    for i, r in S[S.o_que_falta == "ver famílias da empresa no pool"].iterrows():
        if r.aeronave not in an.index: continue
        x = an.loc[r.aeronave]
        S.loc[i, "resposta_pool"] = {"família": "mostrado", "dúvida": "mostrado (dúvida, conta como dentro)", "nenhuma": "nenhuma", "": "nenhuma (em branco no export)"}[x.resposta]
        S.loc[i, "familia_pool_resposta"] = x.familias
    print("respostas do pool lidas de", ans_f[-1].name)
S["resultado"] = np.select(
    [S.motivo_substituicao != "", S.design_nos_626 == "s", S.familia_pool.notna(),
     S.resposta_pool.str.startswith("mostrado"), S.resposta_pool.str.startswith("nenhuma"),
     S.o_que_falta.str.startswith("ver") & (S.n_familias_candidatas == 0)],
    ["substituído", "capturado (nos 626)", "em falta, com patente (família das 51)",
     "em falta, com patente (pool)", "sem patente no pool", "sem patente no pool (nenhuma família candidata)"], "por procurar (query nova)")
S.loc[S.o_que_falta.str.startswith("ver") & (S.resultado == "por procurar (query nova)"), "resultado"] = "por ver na página do pool"
out_cols += ["n_familias_candidatas", "resposta_pool", "familia_pool_resposta", "resultado"]
S[out_cols].to_csv(OUT / "gs_v2_sample.csv", index=False)
print(S[S.motivo_substituicao == ""].resultado.value_counts().to_string())

## 3c. Censo dos programas maduros
Dois critérios externos e verificáveis, sem sorteio, todos os designs entram:
1. **AAM Reality Index (Maio 2026)**: os veículos do índice, que classifica os programas mais avançados para certificação;
2. **VFS com TRL >= 7**: páginas do domínio cuja frase de prova relata um voo tripulado, uma transição para voo em frente, ou um certificado
   (notebooks/02c_vfs_page_trl7.py; a frase fica na tabela para o autor confirmar). TRL 6 não entra: inclui pairar, voo preso por cabo e voo não tripulado.
Filtros comuns: VTOL, eléctrico, tripulado (domínio VFS), não carro voador, actor comercial, primeiro ano >= 2000.

In [ ]:
ix_keys = set(map(tuple, imap.dropna(subset=["aeronave"])[["ck", "mk"]].values))
# Histórico do índice (ari_history.csv, Dez 2020 - Fev 2026): empresas que estiveram no índice e já não estão na edição de Maio 2026.
# O histórico só dá a empresa; o veículo é o programa de passageiros que a empresa tinha no período (escolha do Claude, a validar pelo autor).
INDEX_HIST = {"AIRBUS": [("airbus", "vahana"), ("airbus", "cityairbus"), ("airbus", "cityairbusnextgen")], "BELL": [("bell", "nexus4ex")],
              "PIPISTREL": [("pipistrelaviation", "801")], "LILIUM": [("lilium", "liliumjet")], "OVERAIR": [("overair", "butterfly")],
              "JAUNT AIR MOBILITY": [("jauntairmobility", "journey")], "KITTYHAWK": [("kittyhawk", "heaviside")],
              "VOLKSWAGEN": [("volkswagengroupchina", "vmoflyingtiger")], "WANFENG DIAMOND": [("wanfengautoholdinggroup", "wanfengdiamond")]}
INDEX_HIST_OUT = {"SABREWING AIRCRAFT COMPANY": "Rhaegal: carga, não tripulado, turbo-eléctrico", "ELROY AIR": "Chaparral: carga, não tripulado, híbrido",
                  "DUFOUR AEROSPACE": "Aero2/Aero3: híbrido, fora do domínio no VFS", "AIRFLOW": "eSTOL, não é VTOL", "EVIATION": "Alice: avião convencional, não é VTOL",
                  "eAVIATION": "sem página VFS", "ALAKA'I TECHNOLOGIES": "Skai: hidrogénio, fora do domínio no VFS", "BELL (Nexus 6HX)": "híbrido"}
hist_keys = {k for v_ in INDEX_HIST.values() for k in v_}
miss_h = hist_keys - set(zip(des.ck, des.mk)); assert not miss_h, miss_h
pd.DataFrame([{"oem": o, "resultado": "fora", "motivo": m_} for o, m_ in INDEX_HIST_OUT.items()]).to_csv(OUT / "gs_v2_census_index_history_out.csv", index=False)
D = des.copy()
D["no_indice"] = [(c, m) in ix_keys for c, m in zip(D.ck, D.mk)]
D["trl7_ok"] = D.trl7.fillna(0) >= 7
D["ano_1o_voo"] = D.ck.map(lambda c: None)
D = D.merge(imap.dropna(subset=["aeronave"]).groupby(["ck", "mk"]).ano_indice.min().reset_index(), on=["ck", "mk"], how="left")
D["ano_1o_voo"] = D.ano_indice.fillna(D.ano_vfs)
D["no_hist"] = [(c, m) in hist_keys for c, m in zip(D.ck, D.mk)]
C = D[D.no_indice | D.no_hist | D.trl7_ok].copy()
C["categoria_vfs"] = C.classe.map(CAT)
C["criterio"] = np.select([C.no_indice & C.trl7_ok, C.no_indice, C.no_hist & C.trl7_ok, C.no_hist],
                          ["índice + TRL 7", "índice AAM", "índice (histórico) + TRL 7", "índice AAM (histórico)"], "VFS TRL 7")
def fora(r):
    if not r.dominio_vfs: return "fora do domínio VFS (ex.: Honda F1, híbrido)"
    if r.carro_voador: return "carro voador"
    if ACAD.search(str(r.empresa)) or not r.nome_legal: return "actor académico ou particular"
    if pd.notna(r.ano_1o_voo) and r.ano_1o_voo < 2000: return "anterior a 2000"
    if r.classe not in CAT: return "sem categoria VFS"
    return ""
C["motivo_fora"] = C.apply(fora, axis=1)
C = C.merge(e[["ck", "mk", "familia_pool"]], on=["ck", "mk"], how="left")
C = pd.concat([C, C.apply(cross, axis=1)], axis=1)
an = pd.read_csv(ans_f[-1], dtype=str).fillna("").set_index("aeronave") if ans_f else pd.DataFrame(columns=["resposta", "familias"])
C["resposta_pool"] = C.aeronave.map(lambda a_: an.resposta.get(a_) if a_ in an.index else None)
C["familia_pool_resposta"] = C.aeronave.map(lambda a_: an.familias.get(a_, "") if a_ in an.index else "")
# verificações do autor no census_check (Eve, VE25-100): gs_v2_census_answers.csv, aeronave -> id nos 626 ou "nenhum"
ca = OUT / "gs_v2_census_answers.csv"
cans = pd.read_csv(ca, dtype=str).fillna("").set_index("aeronave").resposta if ca.exists() else pd.Series(dtype=str)
SIMILAR_OTHER = {"Axe"}     # capturas aceites pelo autor com um design semelhante de outro requerente; ficam visíveis no resultado
def res_c(r):
    if r.motivo_fora: return "fora"
    if r.aeronave in cans.index and cans[r.aeronave] not in ("", "nenhum"):
        return "capturado (design semelhante de outra empresa, decisão do autor)" if r.aeronave in SIMILAR_OTHER else "capturado (nos 626)"
    if r.design_nos_626 == "s" and not (r.aeronave in cans.index and cans[r.aeronave] == "nenhum"): return "capturado (nos 626)"
    if isinstance(r.familia_pool, str) and r.familia_pool: return "em falta, com patente (família das 51)"
    if r.resposta_pool in ("família", "dúvida"): return "em falta, com patente (pool)"
    if r.aeronave in cans.index: return "por procurar (query nova)" if r.empresa_no_pool_572 == "n" else "sem patente no pool"
    if r.resposta_pool is not None: return "sem patente no pool"
    if r.empresa_no_pool_572 == "s": return "por ver (pool)"
    return "por procurar (query nova)"
C["resultado"] = C.apply(res_c, axis=1)
# query nova do censo: export em 01_query/patseer_gs_v2_*.xlsx, página feita por notebooks/02d_census_query_page.py,
# respostas do autor em gs_v2_census_query_answers_<stamp>.csv (Transferências ou esta pasta; a mais recente vale)
qd = OUT / "gs_v2_census_query_designs.csv"
qa = sorted(list(OUT.glob("gs_v2_census_query_answers_*.csv")) + list((Path.home() / "Downloads").glob("gs_v2_census_query_answers_*.csv")), key=lambda f: f.stat().st_mtime)
if qd.exists():
    nq = pd.read_csv(qd).set_index("aeronave").n_familias
    qan = pd.read_csv(qa[-1], dtype=str).fillna("").set_index("aeronave") if qa else pd.DataFrame(columns=["resposta", "familias"])
    if qa and qa[-1].parent != OUT: (OUT / qa[-1].name).write_bytes(qa[-1].read_bytes())
    for i, r in C[C.resultado == "por procurar (query nova)"].iterrows():
        if r.aeronave not in nq.index: continue
        ans = qan.resposta.get(r.aeronave, "") if r.aeronave in qan.index else ""
        if nq[r.aeronave] == 0: C.loc[i, "resultado"] = "sem patente (query nova: 0 famílias)"
        elif ans in ("família", "dúvida"): C.loc[i, "resultado"] = "em falta, com patente (query nova)"; C.loc[i, "familia_pool_resposta"] = qan.familias[r.aeronave]
        elif qa: C.loc[i, "resultado"] = "sem patente (query nova: nenhuma família mostra)"
        else: C.loc[i, "resultado"] = "por ver (página da query nova)"
CC = ["criterio", "empresa", "aeronave", "categoria_vfs", "ano_1o_voo", "resultado", "motivo_fora", "frase_trl7", "ids_nos_626",
      "familia_pool", "familia_pool_resposta", "empresa_no_pool_572", "url", "ck", "slugs"]
C = C.sort_values(["motivo_fora", "criterio", "categoria_vfs", "empresa"])
C[CC].to_csv(OUT / "gs_v2_census.csv", index=False)
print(C.groupby("criterio").size().to_string()); print(); print(C.resultado.value_counts().to_string())
IN = C[C.motivo_fora == ""]; cap = int(IN.resultado.str.startswith("capturado").sum()); mis = int(IN.resultado.str.startswith("em falta").sum())
pend = int(IN.resultado.str.startswith("por").sum()); nop = len(IN) - cap - mis - pend
print(f"\ncenso: {len(IN)} programas | capturados {cap} | em falta com patente {mis} | sem patente {nop} | por decidir {pend}")
print(f"completude total (autor, 2026-11-10: todas as aeronaves do censo contam): {cap}/{len(IN)} = {100 * cap / len(IN):.0f} %")
print(f"completude entre as aeronaves com patente encontrada: {cap}/{cap + mis} = {100 * cap / (cap + mis):.0f} %")
C[CC[:7]]

## 4. Queries para as empresas da amostra fora do pool

In [ ]:
# nomes de requerente: o VFS dá o nome comercial; aqui ficam as formas a procurar (rever antes de correr)
# nomes de requerente, entre aspas como na query do gold standard; o VFS dá o nome comercial
ASN_FIX = {"pipistrel": ["Pipistrel"], "wanfeng": ["Wanfeng"], "volant": ["Wollant", "Volant Aerotech"], "zerog": ["Zero Gravity Aircraft", "ZeroG"], "leoflight": ["LEO Flight"],
           "star8green": ["Star 8 Green", "Koncepto"], "tier1engineering": ["Tier 1 Engineering"], "moller": ["Moller International", "Freedom Motors"],
           "avic": ["Aviation Industry Corporation of China", "AVIC", "China Helicopter Research and Development Institute"],
           "pivotal": ["Pivotal", "Opener"], "hongqi": ["Hongqi", "FAW"], "caltech": ["California Institute of Technology"], "tetra": ["teTra Aviation"], "trekaerospace": ["Trek Aerospace"]}
def q_(s): return f'"{s}"' if " " in s or "-" in s else s
def asn_name(s):
    k = ckey(s)
    for pat, v_ in ASN_FIX.items():
        if k.startswith(pat): return " OR ".join(q_(x) for x in v_)
    n = re.sub(r"\(.*?\)", " ", str(s)); n = re.sub(LEGAL, " ", n, flags=re.I)
    n = re.sub(r"[,.]", " ", n); n = re.sub(r"\s+", " ", n).strip()
    return q_(n)
lines = ["# Gold standard v2: queries PatSeer para as empresas da amostra que não estão no pool das 572.",
         "# Forma: ASN:(<empresa>) AND CPC:(B64C* OR B64U*) AND PBY:[<ano_1o_voo-6> TO <ano_1o_voo+2>]",
         "# Não foram corridas. Rever o nome do requerente antes de correr (o VFS dá o nome comercial).", ""]
need = S[S.precisa_procura_nova == "s"]
for ck_, g in need.groupby("ck", sort=False):
    yrs = g.ano_1o_voo.dropna().astype(int)
    designs = "; ".join(f"{a_} ({c_}, {'s/ ano' if pd.isna(y_) else int(y_)})" for a_, c_, y_ in zip(g.aeronave, g.categoria_vfs, g.ano_1o_voo))
    rng_ = f"PBY:[{yrs.min() - 6} TO {yrs.max() + 2}]" if len(yrs) else "PBY:[ANO_EM_FALTA]"
    lines.append(f"ASN:({asn_name(g.empresa.iloc[0])}) AND CPC:(B64C* OR B64U*) AND {rng_}   # {designs}")
clauses = [re.sub(r"\s+#.*$", "", l).replace(" AND CPC:(B64C* OR B64U*)", "") for l in lines if l.startswith("ASN:")]
single = "(" + " OR ".join("(" + c + ")" for c in clauses) + ") AND CPC:(B64C* OR B64U*)"
lines = lines[:4] + ["# QUERY ÚNICA: as %d empresas numa só pesquisa, um só export. Usar esta." % len(clauses), single, "",
                     "# As mesmas queries, uma por empresa (só para referência ou se a query única for longa demais):"] + lines[4:]
(OUT / "gs_v2_queries.txt").write_text("\n".join(lines) + "\n", encoding="utf-8")
print(len(clauses), "empresas | query única:", len(single), "caracteres")

In [ ]:
# query do censo: empresas com designs maduros "por procurar"
need_c = C[C.resultado == "por procurar (query nova)"]
cl = []
for ck_, g in need_c.groupby("ck", sort=False):
    y = g.ano_1o_voo.dropna().astype(int)
    lo, hi = (y.min() - 6, min(y.max() + 2, 2026)) if len(y) else (2000, 2026)
    cl.append(f"(ASN:({asn_name(g.empresa.iloc[0])}) AND PBY:[{lo} TO {hi}])")
    print(f"{g.empresa.iloc[0]:45s} {lo}-{hi}  " + "; ".join(g.aeronave))
qc = "(" + " OR ".join(cl) + ") AND CPC:(B64C* OR B64U*)"
(DATA_DIR / "01_query/query_goldstandard_v2_census.txt").write_text(qc + "\n", encoding="utf-8")
print(len(cl), "empresas;", len(qc), "caracteres")

## 4c. O filtro do estado da página enviesa a arquitectura?

In [ ]:
x = v[v.in_domain]
dist = pd.DataFrame({"páginas no domínio": x.evtol_class_main.value_counts(), "com hardware (estado)": x[x.status_group == "flying hardware"].evtol_class_main.value_counts(),
                     "sem estado": x[x.status_group == "not stated"].evtol_class_main.value_counts(), "designs elegíveis": elig.classe.value_counts(),
                     "empresas elegíveis": elig.groupby("classe").ck.nunique()}).reindex([k for k, _, _ in QUOTA]).fillna(0).astype(int)
dist["% do domínio"] = (100 * dist["páginas no domínio"] / dist["páginas no domínio"].sum()).round(1)
dist["% dos elegíveis"] = (100 * dist["designs elegíveis"] / dist["designs elegíveis"].sum()).round(1)
dist.index = [CAT[k] for k in dist.index]
dist

## 5. Resultado

In [ ]:
print("Elegíveis por categoria"); display(tab)
show = S[["categoria_vfs", "ordem_sorteio", "empresa", "aeronave", "trl", "ano_1o_voo", "origem", "gold_v1_patentes_do_design",
          "gold_v1_patentes_da_empresa", "empresa_no_pool_572", "design_nos_626", "familia_pool", "substituido_por"]].copy()
show["empresa"] = show.empresa.str.slice(0, 34); show["familia_pool"] = show.familia_pool.fillna("").str.slice(0, 30)
display(show)
A_ = S[S.motivo_substituicao == ""]
print("linhas:", len(S), "| designs activos:", len(A_), "| substituídos:", len(S) - len(A_))
print("empresa fora do pool -> procura nova:", int((S.precisa_procura_nova == "s").sum()), "designs,", need.ck.nunique(), "empresas")
print("design já nos 626:", int((A_.design_nos_626 == "s").sum()), "| com patente no gold v1:", int((A_.gold_v1_patentes_do_design != "").sum()), "| vindos das 51:", int(A_.familia_pool.notna().sum()))